# YOLO11l Object Detection Exploration

This notebook explores the pretrained **YOLO11l** object-detection model with Ultralytics. It loads an image, runs inference, inspects detections, visualizes bounding boxes, and saves an annotated result.

> The first model run may download `yolo11l.pt` and the sample image if they are not already cached.

In [ ]:
# Install dependencies from a notebook cell when needed.
%pip install -q ultralytics matplotlib pandas pillow

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
VEHICLE_DIR = NOTEBOOKS_DIR / "vehicle_detection"
DATA_DIR = VEHICLE_DIR / "data"
OUTPUT_DIR = VEHICLE_DIR / "outputs"
MODEL_PATH = VEHICLE_DIR / "yolo11l.pt"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "yolo11l.pt"
IMAGE_PATH = DATA_DIR / "auto.JPG"
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Place the vehicle image at {IMAGE_PATH}")

print(f"Vehicle detection directory: {VEHICLE_DIR}")
print(f"Input image: {IMAGE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load the pretrained large YOLO11 model.
model = YOLO(str(MODEL_PATH) if MODEL_PATH.exists() else MODEL_NAME)
print(model)

In [ ]:
# Preview the input image.
image = Image.open(IMAGE_PATH)
plt.figure(figsize=(12, 8))
plt.imshow(image)
plt.axis("off")
plt.title(f"Input image: {image.size[0]} x {image.size[1]}")
plt.show()

In [ ]:
# Run object detection. Adjust confidence and image size to explore the trade-off
# between detection sensitivity, speed, and memory usage.
results = model.predict(
    source=str(IMAGE_PATH),
    conf=0.55,
    iou=0.7,
    imgsz=640,
    device="cpu",  # Change to 0 for the first CUDA GPU.
    verbose=True,
)
result = results[0]
print(f"Detected {len(result.boxes)} objects")

In [ ]:
# Display the annotated image produced by Ultralytics.
annotated = result.plot()
plt.figure(figsize=(14, 9))
plt.imshow(annotated[..., ::-1])  # BGR -> RGB
plt.axis("off")
plt.title("YOLO11l detections")
plt.show()

In [ ]:
# Convert detections into a table for inspection or downstream analysis.
boxes = result.boxes
detection_table = pd.DataFrame({
    "class_id": boxes.cls.cpu().numpy().astype(int),
    "class_name": [result.names[int(class_id)] for class_id in boxes.cls.cpu().numpy()],
    "confidence": boxes.conf.cpu().numpy(),
    "x1": boxes.xyxy[:, 0].cpu().numpy(),
    "y1": boxes.xyxy[:, 1].cpu().numpy(),
    "x2": boxes.xyxy[:, 2].cpu().numpy(),
    "y2": boxes.xyxy[:, 3].cpu().numpy(),
})
detection_table.sort_values("confidence", ascending=False).reset_index(drop=True)

In [ ]:
# Save the annotated image and export detections to CSV.
annotated_path = OUTPUT_DIR / "auto_yolo11l_annotated.jpg"
csv_path = OUTPUT_DIR / "auto_yolo11l_detections.csv"
result.save(filename=str(annotated_path))
detection_table.to_csv(csv_path, index=False)
print(f"Saved annotated image: {annotated_path}")
print(f"Saved detections CSV: {csv_path}")

## Experiments

Try changing one variable at a time:

- Set `device=0` if CUDA is available.
- Compare `conf=0.1`, `0.25`, and `0.5`.
- Compare `imgsz=640` and `imgsz=1280`.
- Replace `IMAGE_PATH` with an image from your own dataset.
- Use `model.track(...)` on a video if you want to extend this exploration to object tracking.